# 02 · Solver backends and the staircase

Two backends solve the same χ² problem:

- `scipy` — `trust-constr`, stable, the safe default.
- `osqp` — formulates χ² as a QP; fast. For this under-determined
  problem (180 params, 29 bins) it additionally runs a **simplex LP** to
  pick a *vertex* — a piecewise-constant (staircase) flux.

Both report the same χ². The flux *shape* can differ because the optimum
is a whole face, not a point.

In [ ]:
import warnings; warnings.simplefilter('ignore')
from neutrino_analysis_band import NeutrinoAnalysis
import numpy as np

def flat_pairs(x, tol=1e-5):
    d = np.abs(np.diff(x))
    return int(np.sum(d <= tol * np.abs(x).max()))

a = NeutrinoAnalysis(background_scenario='flat', intervals='180',
                     GeV=0.32e16, solver='scipy', T=3)
xs = a.optimize(a.data_vector).x
print('scipy : chi2/c=%.2e  flat-adjacent pairs=%d/%d'
      % (a.result.fun / a.c, flat_pairs(xs), a.n - 1))

In [ ]:
# Switch to OSQP on the same object.
a.set_solver('osqp')
xo = a.optimize(a.data_vector).x
print('osqp  : chi2/c=%.2e  flat-adjacent pairs=%d/%d'
      % (a.result.fun / a.c, flat_pairs(xo), a.n - 1))
print('osqp staircase? (many flat pairs ->) ', flat_pairs(xo) > 100)

In [ ]:
# Sanity: the returned flux really achieves the reported chi2.
d, b, M = a.data_vector, a.Bkg_vector, a.M_matrix
resid = (d - b) - M @ xo
inv = np.where(d > 0, 1 / np.where(d > 0, d, 1), 0)
chi2_hand = a.T * np.sum(resid * resid * inv) / a.c
print('reported chi2/c = %.3e' % (a.result.fun / a.c))
print('hand-computed   = %.3e' % chi2_hand)

**Vertex selection** is controlled by `a._backend.vertex_select`
(default `True`). Set it to `False` to get the smooth interior solution.
The staircase needs a simplex LP solver; the code tries HiGHS → GLPK →
SciPy, and SciPy ships with cvxpy, so it works out of the box.